<a href="https://colab.research.google.com/github/jaewon00o0/kbu-2.1/blob/main/5%EC%9B%94_%EC%A4%91%EA%B0%84%EA%B3%A0%EC%82%AC_%EC%95%B1%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok beautifulsoup4 httpx gradio pandas matplotlib

In [ ]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3ChPQkgS0bfV4YFKDyyMtr0tSim_6WkYoSaYtNJKY2XVdqxE2"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)



In [ ]:
import sqlite3
import httpx
from bs4 import BeautifulSoup
from fastapi import FastAPI
from pydantic import BaseModel
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import io
from PIL import Image
import nest_asyncio
import uvicorn
from pyngrok import ngrok

1. 데이터베이스 초기화 및 크롤링

In [ ]:
DB_FILE = "quotes.db"

def init_db():
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS quotes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            text TEXT NOT NULL,
            author TEXT NOT NULL,
            category TEXT NOT NULL
        )
    ''')
    conn.commit()
    conn.close()

def crawl_and_save_quotes(category="love", target_count=20):
    """지정된 개수를 채울 때까지 페이지를 넘기며 크롤링"""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    count = 0
    page = 1

    while count < target_count:
        url = f"https://quotes.toscrape.com/tag/{category}/page/{page}/"
        response = httpx.get(url)
        soup = BeautifulSoup(response.text, "html.parser")
        quotes = soup.find_all("div", class_="quote")

        if not quotes:
            break

        for q in quotes:
            if count >= target_count:
                break

            text = q.find("span", class_="text").text
            author = q.find("small", class_="author").text

            cursor.execute("SELECT id FROM quotes WHERE text=?", (text,))
            if not cursor.fetchone():
                cursor.execute("INSERT INTO quotes (text, author, category) VALUES (?, ?, ?)",
                               (text, author, category))
                count += 1
        page += 1

    conn.commit()
    conn.close()
    print(f" [{category}] 카테고리 명언 {count}개 수집 완료")

init_db()
crawl_and_save_quotes("love", 20)
crawl_and_save_quotes("life", 20)

 [love] 카테고리 명언 0개 수집 완료
 [life] 카테고리 명언 0개 수집 완료


FastAPI 백엔드

In [ ]:
app = FastAPI(title="명언 관리 시스템 API")

class QuoteCreate(BaseModel):
    text: str
    author: str
    category: str

class QuoteUpdate(BaseModel):
    text: str = None
    author: str = None
    category: str = None

@app.post("/api/quotes/", summary="명언 추가")
def create_quote(quote: QuoteCreate):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("INSERT INTO quotes (text, author, category) VALUES (?, ?, ?)",
                   (quote.text, quote.author, quote.category))
    conn.commit()
    new_id = cursor.lastrowid
    conn.close()
    return {"id": new_id, "message": "성공적으로 추가되었습니다."}

@app.get("/api/quotes/", summary="명언 목록 조회")
def read_quotes():
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("SELECT id, text, author, category FROM quotes")
    rows = cursor.fetchall()
    conn.close()
    return [{"id": r[0], "text": r[1], "author": r[2], "category": r[3]} for r in rows]

3. Gradio 프론트엔드 및 데이터 분석

In [ ]:
def get_word_count_plot():
    conn = sqlite3.connect(DB_FILE)
    df = pd.read_sql_query("SELECT text FROM quotes", conn)
    conn.close()

    if df.empty:
        return None

    words = df['text'].str.replace('[^\w\s]', '', regex=True).str.lower().str.split().sum()
    word_counts = pd.Series(words).value_counts().head(10)

    fig, ax = plt.subplots(figsize=(10, 5))
    word_counts.plot(kind='bar', ax=ax, color='coral')
    ax.set_title("Top 10 Most Frequent Words")
    ax.set_xlabel("Words")
    ax.set_ylabel("Frequency")

    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    plt.close(fig)
    return Image.open(buf)

with gr.Blocks() as gradio_app:
    gr.Markdown("# 코랩 기반 격언 관리 및 분석 시스템")

    with gr.Tab("데이터 분석"):
        gr.Markdown("### 단어 빈도수 분석")
        plot_output = gr.Image(type="pil")
        btn_analyze = gr.Button("단어 빈도수 시각화하기")
        btn_analyze.click(fn=get_word_count_plot, outputs=plot_output)

app = gr.mount_gradio_app(app, gradio_app, path="/ui")

<>:9: SyntaxWarning: invalid escape sequence '\w'
<>:9: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_14169/2102629505.py:9: SyntaxWarning: invalid escape sequence '\w'
  words = df['text'].str.replace('[^\w\s]', '', regex=True).str.lower().str.split().sum()


new /ui


4. ngrok 외부 터널링 및 Uvicorn 서버 실행

In [ ]:
ngrok.kill()

In [ ]:
public_url = ngrok.connect(8000).public_url

In [ ]:
public_url = ngrok.connect(8000).public_url

print("\n" + "="*60)
print(f"API 명세서 (Swagger UI): {public_url}/docs")
print(f"프론트엔드 UI (Gradio): {public_url}/ui")
print("="*60 + "\n")


API 명세서 (Swagger UI): https://napkin-hamster-ladies.ngrok-free.dev/docs
프론트엔드 UI (Gradio): https://napkin-hamster-ladies.ngrok-free.dev/ui



In [ ]:
nest_asyncio.apply()

In [ ]:
import nest_asyncio
import uvicorn
import asyncio

nest_asyncio.apply()

async def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    await server.serve()

if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    loop.create_task(run_server())